# Prices, Returns & Drawdowns

**Markets · Beginner · ~10 min · pandas + yfinance · needs an internet connection**

Download three years of real daily prices for one symbol from Yahoo Finance (through the `yfinance` package) and look at them the way an analyst would: the trend, the size of a normal day, how deep the falls have been, and a one-table summary of return and risk. Change the symbol in the parameter cell and run all cells again to study any stock, ETF, index or crypto pair that Yahoo covers. The notebook needs an internet connection and writes nothing to disk.

**What you will see**
- A table of the last 10 trading days (open, high, low, close, volume)
- A price chart with the 50- and 200-day moving averages
- A histogram of daily returns next to a normal (bell) curve
- A drawdown chart: how far the price sat below its previous peak
- A summary table: CAGR, annualised volatility, Sharpe ratio, max drawdown, best and worst day

## 1. Parameters

Everything you might want to change lives in this one cell. `SYMBOL` is any Yahoo Finance ticker. Keep `PERIOD` longer than the slow moving average (200 trading days is about ten months) so that average has data to draw. Crypto trades every day of the year, so set `TRADING_DAYS = 365` for a pair such as `BTC-USD`.

In [ ]:
# ── Parameters: edit these, then Run All (Ctrl+Shift+Enter) ──────────
SYMBOL = "AAPL"        # e.g. "MSFT", "SPY", "^GSPC" (S&P 500 index), "BTC-USD", "RELIANCE.NS"
PERIOD = "3y"          # history to download: "1y", "2y", "3y", "5y", "10y" or "max"
FAST_MA = 50           # fast moving average, in trading days
SLOW_MA = 200          # slow moving average, in trading days
TRADING_DAYS = 252     # trading days per year, used to annualise (365 for crypto)

## 2. Imports and chart style

`yfinance` downloads the prices, pandas holds them in a table indexed by date, and matplotlib draws the charts. The `rcParams` block only gives every chart in this notebook the same clean look.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# One consistent, quiet chart style for every figure in this notebook
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "lines.linewidth": 1.6, "legend.frameon": False,
})

## 3. Download daily prices

`yf.download` returns one row per trading day. With `auto_adjust=True` the prices are adjusted for splits and dividends, so the change from one close to the next is the true total return. yfinance labels the columns with *(field, ticker)* pairs even for a single symbol, so we drop the ticker level to get plain `Open / High / Low / Close / Volume` columns.

The cell ends with a DataFrame, so the last 10 rows appear as a sortable table.

In [ ]:
data = yf.download(SYMBOL, period=PERIOD, auto_adjust=True, progress=False)
if data.empty:
    raise ValueError(f"No price data for {SYMBOL} - check the symbol or your connection")

# Columns arrive as (field, ticker) pairs even for one symbol: keep only the field names
prices = data.droplevel("Ticker", axis=1)[["Open", "High", "Low", "Close", "Volume"]].dropna()
close = prices["Close"]

print(f"{SYMBOL}: {len(prices)} trading days, "
      f"{prices.index[0]:%Y-%m-%d} to {prices.index[-1]:%Y-%m-%d}")
prices.tail(10).round(2)

## 4. Price with 50- and 200-day moving averages

A moving average is the mean of the last *N* closes, recomputed every day: `close.rolling(N).mean()`. The 200-day average is the classic long-term trend line. Price above it means an uptrend by this measure, below it a downtrend. The 50-day line crossing above or below the 200-day line is the well-known *golden cross* / *death cross*. An N-day average needs N days of history, so those lines start later than the price.

In [ ]:
ma_fast = close.rolling(FAST_MA).mean()
ma_slow = close.rolling(SLOW_MA).mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(close, label=f"{SYMBOL} close", linewidth=1.2)
ax.plot(ma_fast, label=f"{FAST_MA}-day average")
ax.plot(ma_slow, label=f"{SLOW_MA}-day average")
ax.set_title(f"{SYMBOL}: daily close and moving averages")
ax.set_xlabel("Date")
ax.set_ylabel("Price (split & dividend adjusted)")
ax.legend(loc="upper left")
plt.tight_layout()

if ma_slow.notna().any():
    side = "above" if close.iloc[-1] > ma_slow.iloc[-1] else "below"
    print(f"Last close {close.iloc[-1]:,.2f} is {side} its {SLOW_MA}-day average ({ma_slow.iloc[-1]:,.2f}).")
else:
    print(f"Not enough history for a {SLOW_MA}-day average - use a longer PERIOD.")

## 5. How big is a normal day? The return distribution

The daily return is today's close divided by yesterday's, minus one (`pct_change()`). The histogram shows how often each size of move happened. The orange line is a normal (bell-curve) distribution with the same mean and volatility. Real returns usually have a taller middle and **fatter tails**: more quiet days *and* more extreme days than the bell curve predicts. *Excess kurtosis* measures exactly that; it is 0 for a normal distribution.

In [ ]:
returns = close.pct_change().dropna()
pct = returns * 100                      # in percent, for readable axes

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pct, bins=60, density=True, alpha=0.85, edgecolor="white", linewidth=0.4,
        label="Daily returns")
# Normal curve with the same mean and standard deviation, for comparison
mu, sd = pct.mean(), pct.std()
x = np.linspace(pct.min(), pct.max(), 300)
ax.plot(x, np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)),
        color="#eb6834", label="Normal curve, same mean & volatility")
ax.axvline(0, color="#898781", linewidth=0.8)
ax.set_title(f"{SYMBOL}: distribution of daily returns")
ax.set_xlabel("Daily return (%)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()

print(f"Average day {mu:+.2f}%, typical move (std) {sd:.2f}%, "
      f"skew {returns.skew():.2f}, excess kurtosis {returns.kurt():.2f}")
print(f"Days that moved more than 3%: {(pct.abs() > 3).sum()} of {len(pct)}")

## 6. Drawdowns: the chart investors actually live through

A drawdown is how far the price is below the highest close seen so far: `close / close.cummax() - 1`. Zero means a new high; −30% means anyone who bought at the top is down 30%. The deepest point is the **maximum drawdown**, one of the most important risk numbers because it is the loss an investor has to sit through.

In [ ]:
drawdown = close / close.cummax() - 1
trough = drawdown.idxmin()                    # date of the deepest point
peak = close.loc[:trough].idxmax()            # the high before it
after = drawdown.loc[trough:]
recovered = after[after >= 0].index.min()     # first new high after the trough (NaT if none)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.fill_between(drawdown.index, drawdown * 100, 0, alpha=0.3, linewidth=0)
ax.plot(drawdown * 100, linewidth=1)
ax.annotate(f"max drawdown {drawdown.min():.1%}", xy=(trough, drawdown.min() * 100),
            xytext=(8, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_title(f"{SYMBOL}: drawdown from previous peak")
ax.set_xlabel("Date")
ax.set_ylabel("Below previous peak (%)")
plt.tight_layout()

status = f"recovered on {recovered:%Y-%m-%d}" if pd.notna(recovered) else "not recovered yet"
print(f"Worst fall: {drawdown.min():.1%} from {peak:%Y-%m-%d} to {trough:%Y-%m-%d}, {status}.")

## 7. Summary statistics

- **Total return** and **CAGR** (compound annual growth rate): the steady yearly return that turns the first close into the last one.
- **Annualised volatility**: the standard deviation of daily returns × √252, the usual measure of risk.
- **Sharpe ratio**: annualised mean return ÷ annualised volatility, with the risk-free rate set to 0 for simplicity.
- **Max drawdown**, **best day** and **worst day** come from the cells above.

The result is a DataFrame, so it shows as a table you can sort and copy.

In [ ]:
years = (close.index[-1] - close.index[0]).days / 365.25
total_return = close.iloc[-1] / close.iloc[0] - 1
cagr = (1 + total_return) ** (1 / years) - 1
ann_vol = returns.std() * np.sqrt(TRADING_DAYS)
sharpe = returns.mean() / returns.std() * np.sqrt(TRADING_DAYS)   # risk-free rate = 0

summary = pd.Series({
    "Total return (%)": total_return * 100,
    "CAGR (%)": cagr * 100,
    "Annualised volatility (%)": ann_vol * 100,
    "Sharpe ratio (rf = 0)": sharpe,
    "Max drawdown (%)": drawdown.min() * 100,
    f"Best day, {returns.idxmax():%Y-%m-%d} (%)": returns.max() * 100,
    f"Worst day, {returns.idxmin():%Y-%m-%d} (%)": returns.min() * 100,
}, name=SYMBOL)
summary.index.name = "Statistic"

print(f"{SYMBOL}, {close.index[0]:%Y-%m-%d} to {close.index[-1]:%Y-%m-%d} ({years:.1f} years)")
summary.round(2).to_frame()

## Try next

- Change `SYMBOL` to an index (`^GSPC`), an ETF (`QQQ`) or a crypto pair (`BTC-USD` with `TRADING_DAYS = 365`) and run all cells again.
- Set `PERIOD = "10y"` to see how the drawdown chart changes over a full market cycle.
- Open **Portfolio Returns & Risk vs a Benchmark** to combine several holdings, or **Trend-Following Backtest on Real Prices** to trade the moving-average signal.

---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*